# Inteligência Artificial em Jogos 25/26
## Projeto NEAT - Versão 1: Sensores (Radares)

**Introdução e Arquitetura**
Este notebook implementa a abordagem baseada em radares exigida no **Requisito 3.2** do enunciado. Ao contrário dos sensores de tamanho fixo usados nas aulas de laboratório, este controlador utiliza *Raycasting* para lançar raios virtuais a partir do centro do carro, medindo a distância até à borda da pista.

Para manter o código limpo e modular, toda a lógica base do Pygame, classes genéricas e carregamento de texturas foi movida para o ficheiro partilhado `base_jogo.py`. Aqui focamo-nos estritamente na Neuro-evolução (NEAT).

In [1]:
from base_jogo import *

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


### Classe do Carro (NeatCar) e Raycasting

Nesta secção definimos a classe `NeatCar` que herda as propriedades físicas do `AbstractCar`. As principais inovações desta classe são:

1. **Inputs (Sensores):** São lançados 5 radares simétricos (-60º, -30º, 0º, 30º, 60º). Os valores são **normalizados** (entre 0.0 e 1.0) dividindo a distância percorrida pelo raio pelo limite máximo de visão (200px), facilitando a convergência da rede neuronal.
2. **Outputs (Atuadores):** A rede controla os valores contínuos de aceleração (*throttle*) e rotação (*steering*).
3. **Ruído Mecânico (Requisito 3.3):** Adicionámos uma pequena incerteza aleatória (`uniform(-0.05, 0.05)`) aos comandos injetados pela rede, simulando o comportamento não determinístico do mundo físico.

In [ ]:
class NeatCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)

    def __init__(self, max_vel, rotation_vel, start_angle=0, track_reversed=False):
        super().__init__(max_vel, rotation_vel)
        self.angle = start_angle
        self.track_reversed = track_reversed
        if self.track_reversed:
            self.x, self.y = (175, 270) 
        else:
            self.x, self.y = (150, 200)
        self.alive = True
        self.distance = 0
        self.radars = []

        self.finished = False 
        self.finish_time = 0

    def check_radar(self, degree, track_border_mask):
        length = 0
        x = int(self.x + CAR_SIZE[0])
        y = int(self.y + CAR_SIZE[1])

        rad = math.radians(self.angle + degree)
        dx = -math.sin(rad)
        dy = -math.cos(rad)

        MAX_RADAR_LENGTH = 200 

        while length < MAX_RADAR_LENGTH:
            x = int(self.x + CAR_SIZE[0] + (dx * length))
            y = int(self.y + CAR_SIZE[1] + (dy * length))

            if x < 0 or x >= WIDTH or y < 0 or y >= HEIGHT:
                break
            
            if track_border_mask.get_at((x, y)):
                break
                
            length += 1

        dist = int(math.sqrt(math.pow(x - (self.x + CAR_SIZE[0]), 2) + math.pow(y - (self.y + CAR_SIZE[1]), 2)))
        self.radars.append([(x, y), dist])

    def update_radars(self):
        """
        Limpa os radares antigos e lança 5 novos em ângulos diferentes.
        """
        self.radars.clear()
        for degree in [-60, -30, 0, 30, 60]:
            self.check_radar(degree, TRACK_BORDER_MASK)

    def get_data(self):
        """
        Retorna os inputs normalizados para a Rede Neuronal.
        """
        return_values = [radar[1] / 200 for radar in self.radars]
        return return_values

    def draw_radars(self, win):
        """
        Desenha as linhas dos radares para poderes ver o carro a pensar.
        """
        for radar in self.radars:
            position = radar[0]
            pygame.draw.line(win, (0, 255, 0), (int(self.x + CAR_SIZE[0]), int(self.y + CAR_SIZE[1])), position, 1)
            pygame.draw.circle(win, (0, 255, 0), position, 3)

    def apply_nn_actions(self, throttle, steering):
        """
        Recebe os outputs da rede neuronal.
        throttle: float entre -1 (travar) e 1 (acelerar no máximo)
        steering: float entre -1 (direita) e 1 (esquerda)
        """
        noise_throttle = uniform(-0.05, 0.05)
        noise_steering = uniform(-0.1, 0.1)
        
        throttle = max(-1.0, min(1.0, throttle + noise_throttle))
        steering = max(-1.0, min(1.0, steering + noise_steering))

        if throttle > 0:
            self.vel = min(self.vel + self.acceleration * throttle, self.max_vel)
        else:
            self.vel = max(self.vel + (self.acceleration * 2) * throttle, 0)

        self.angle += self.rotation_vel * steering
        self.angle = int(self.angle) % 360

        self.move()
        
        self.distance += self.vel 

    def check_collision(self, frame_count):
        if self.collide(TRACK_BORDER_MASK) != None:
            self.alive = False
            self.vel = 0

        if frame_count > 180: 
            finish_poi_collide = self.collide(FINISH_MASK, *FINISH_POSITION)
            if finish_poi_collide != None:
                contra_mao = (finish_poi_collide[1] == 0)
                
                if self.track_reversed:
                    contra_mao = not contra_mao

                if contra_mao:
                    self.alive = False
                    self.vel = 0
                else:
                    self.finished = True
                    self.vel = 0
                    self.finish_time = frame_count / FPS

    def draw(self, win):
        super().draw(win)
        self.draw_radars(win)

### Função de Treino, Checkpoints e Avaliação de Fitness

A função `eval_genomes_radares` treina a população de carros. Para cumprir os **Requisitos 3.4 e 4** do enunciado, implementámos as seguintes lógicas:

* **Inversão de Pista:** A flag `INVERTER_PISTA` permite treinar a rede a conduzir no sentido dos ponteiros do relógio, alterando o ângulo inicial e o *spawn*.
* **Função de Fitness:** A recompensa principal (além da sobrevivência) baseia-se na velocidade instantânea do carro (`ge[i].fitness += car.vel`). Se a rede travar desnecessariamente ou fizer "piões", perde pontos potenciais.
* **Estabilizador de Direção:** Omitimos *drifts* minúsculos (volante com valor absoluto inferior a 0.2) para permitir trajetórias puramente retas e maior velocidade máxima.
* **Checkpoint Anti-Pião:** Carros que não atinjam uma distância de segurança nos primeiros 3 segundos são eliminados por inatividade.

In [ ]:
def eval_genomes_radares(genomes, config):

    INVERTER_PISTA = False 
    ANGULO_INICIAL = 180 if INVERTER_PISTA else 0
    
    nets = []
    cars = []
    ge = []

    for genome_id, genome in genomes:
        genome.fitness = 0 
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        nets.append(net)
        cars.append(NeatCar(max_vel=4, rotation_vel=4, start_angle=ANGULO_INICIAL, track_reversed=INVERTER_PISTA))
        ge.append(genome)

    clock = pygame.time.Clock()
    run = True
    frame_count = 0

    while run and len(cars) > 0:
        clock.tick(FPS)
        frame_count += 1

        if frame_count > 20000:
            break
            
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

        WIN.blit(GRASS, (0, 0))
        WIN.blit(TRACK, (0, 0))
        WIN.blit(FINISH, FINISH_POSITION)

        for i in reversed(range(len(cars))):
            car = cars[i]
            
            car.update_radars()
            inputs = car.get_data() 
            output = nets[i].activate(inputs)
            
            throttle = output[0]
            steering = output[1]

            if abs(steering) < 0.2:
                steering = 0
            
            car.apply_nn_actions(throttle, steering)
            car.check_collision(frame_count)

            if INVERTER_PISTA:
                if car.y < 220 and car.x < 200 and car.x > 150:
                    car.alive = False

            if frame_count == 180:
                dist_start = math.hypot(car.x - 175, car.y - 270) if INVERTER_PISTA else math.hypot(car.x - 150, car.y - 200)
                if dist_start < 100:
                    car.alive = False

            if car.finished:
                ge[i].fitness += 100000  
                
                cars.clear()
                break
                
            elif not car.alive or (car.vel <= 0 and frame_count > 60):
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)
                
            else:
                ge[i].fitness += car.vel
                car.draw(WIN)

        pygame.display.update()

### Execução e Salvaguarda do Modelo
Nesta célula corremos o algoritmo NEAT utilizando os hiperparâmetros definidos em `config-radares.txt`. 

Caso o carro atinja a meta (ganhando um bónus massivo de *fitness* que desencadeia a vitória), o ciclo é interrompido e a melhor rede gerada é exportada usando o módulo `pickle` (`winner_radares.pkl`). Adicionalmente, guardamos os metadados da corrida num ficheiro de estatísticas.

In [ ]:
def run_neat_radares(config_path):
    pygame.init()
    global WIN
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Racing Game - Radares!")

    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                                neat.DefaultSpeciesSet, neat.DefaultStagnation,
                                config_path)

    p = neat.Population(config)

    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    import time
    start_time = time.time()
    winner = p.run(eval_genomes_radares, 25)
    elapsed_time = time.time() - start_time

    pygame.quit()

    print('\nMelhor genoma encontrado:\n{!s}'.format(winner))

    try:
        import pickle
        
        with open('models/winner_radares.pkl', 'wb') as f:
            pickle.dump(winner, f)
        print("\n[OK] Ficheiro 'models/winner_radares.pkl' criado com sucesso!")

        info = {
            'melhor_fitness': winner.fitness,
            'melhor_geracao': p.generation, 
            'tempo_total': elapsed_time,
            'config': config_path,
            'historico_melhor': [c.fitness for c in stats.most_fit_genomes],
            'historico_media': stats.get_fitness_mean()
        }
        
        with open('models/stats_radares.pkl', 'wb') as f:
            pickle.dump(info, f)
        print("[OK] Ficheiro 'models/stats_radares.pkl' criado com sucesso!")
        
        print(f"\nResumo: Meta atingida na geração {p.generation} em {elapsed_time:.1f} segundos.")

    except Exception as e:
        print(f"\n[ERRO AO GUARDAR FICHEIROS]: {e}")

run_neat_radares("configs/config-radares.txt")


 ****** Running generation 0 ****** 

Population's average fitness: 59.29300 stdev: 191.47493
Best fitness: 1401.83251 - size: (2, 10) - species 1 - id 29
Average adjusted fitness: 0.042
Mean genetic distance 2.115, standard deviation 0.488
Population of 120 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0   120   1401.833    0.042     0
Total extinctions: 0
Generation time: 6.276 sec

 ****** Running generation 1 ****** 

Population's average fitness: 67.96526 stdev: 146.24577
Best fitness: 1381.79193 - size: (2, 10) - species 1 - id 29
Average adjusted fitness: 0.049
Mean genetic distance 2.269, standard deviation 0.489
Population of 120 members in 2 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    1    93   1381.792    0.049     1
     2    0    27         --       --     0
Total extinctions: 0
Generation time: 